In [ ]:
import os
import glob
import cv2
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, welch, detrend
 
DATA_DIR = "data"            # contains 'real' and 'fake' subfolders
MAX_SECONDS = 10             # only analyse the first 10 s of each video
LOW_HZ, HIGH_HZ = 0.7, 4.0   # human pulse band (42 - 240 bpm)
MIN_BPM, MAX_BPM = 42, 180   # physiologically plausible heart rate
MIN_FRAMES = 90              # ignore videos that are too short
 
# Landmark indices of MediaPipe Face Mesh used to build skin ROIs
LEFT_CHEEK = [116, 117, 118, 101, 36, 205, 187, 123]
RIGHT_CHEEK = [345, 346, 347, 330, 266, 425, 411, 352]
FOREHEAD_TOP, FOREHEAD_BOTTOM = 10, 9      # top of face / between eyebrows
FOREHEAD_LEFT, FOREHEAD_RIGHT = 103, 332
 
face_mesh = mp.solutions.face_mesh.FaceMesh(
    static_image_mode=False, max_num_faces=1, refine_landmarks=False,
    min_detection_confidence=0.5, min_tracking_confidence=0.5)